In [8]:
!uv pip install langgraph

Using Python 3.12.0 environment at: D:\code\Collection_of_virtual\langchanin_spcae
Checked 1 package in 9ms


In [ ]:
import re
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", temperature=0)

@tool
def get_vehicle_status(query: str = "") -> str:
    """차량의 현재 속도와 연료 상태를 확인합니다."""
    return "현재 속도: 280km/h, 연료: 12.5L"

def save_history(query: str = "") -> str:
    """과거 대화 내용을 찾을수 있습니다."""
    
    return "현재 속도: 280km/h, 연료: 12.5L"


tools = {"get_vehicle_status": get_vehicle_status}

def run_onboard_agent(user_input):
    prompt = f"""
    Answer the user's question. You can use the following tool:
    [get_vehicle_status]

    If you need to use the tool, reply EXACTLY like this:
    Action: get_vehicle_status

    If you have the answer, reply EXACTLY like this:
    Final Answer: [your answer]

    you speak korean

    Question: {user_input}
    """
    
    for _ in range(3):
        response = llm.invoke(prompt).content
        
        if "Final Answer:" in response:
            return response.split("Final Answer:")[-1].strip()
            
        if "Action: get_vehicle_status" in response:
            observation = tools["get_vehicle_status"].invoke({"query": ""})
            prompt += f"\n{response}\nObservation: {observation}\nNow provide the Final Answer:\n"
        else:
            return response.strip()

    return "결과 도출 실패"

print(run_onboard_agent("현재속도"))

280km/h


In [42]:
import json
import re
from typing import Dict, Callable, Any
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

# =========================
# LLM 설정
# =========================
llm = ChatOllama(
    model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M",
    temperature=0
)

# =========================
# Tool 정의
# =========================
@tool
def get_vehicle_status(query: str = "") -> str:
    """차량의 현재 속도와 연료 상태"""
    return "현재 속도: 280km/h, 연료: 12.5L"


@tool
def save_history(query: str = "") -> str:
    """대화 기록 조회"""
    return "히스토리 기능 준비중"


# =========================
# Tool Registry
# =========================
TOOLS: Dict[str, Callable] = {
    "get_vehicle_status": get_vehicle_status,
    "save_history": save_history,
}

# =========================
# JSON 추출 (robust)
# =========================
def extract_json(text: str) -> Dict[str, Any]:
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except:
        pass
    return {}

# =========================
# 프롬프트
# =========================
def build_prompt(user_input: str):
    return f"""
너는 차량 온보드 AI다.

반드시 JSON 형식으로만 답변해라.

형식:
{{
  "action": "tool_name 또는 null",
  "arguments": {{ }},
  "final_answer": "사용자에게 보여줄 답"
}}

규칙:
- tool이 필요하면 action에 tool 이름 작성
- 필요 없으면 action=null
- final_answer는 항상 채워라
- 한국어로 답변

사용 가능한 tool:
- get_vehicle_status: 차량 상태 조회
- save_history: 대화 기록

User: {user_input}
"""

# =========================
# Agent 실행 (핵심)
# =========================
def run_agent(user_input: str):
    prompt = build_prompt(user_input)

    # 🔥 1회 호출
    response = llm.invoke(prompt).content
    print("\n[LLM RAW]")
    print(response)

    parsed = extract_json(response)

    if not parsed:
        return "❌ JSON 파싱 실패"

    action = parsed.get("action")
    final_answer = parsed.get("final_answer", "")

    # =========================
    # tool 실행 필요 없음 → 바로 반환
    # =========================
    if not action or action == "null":
        return final_answer

    # =========================
    # tool 실행
    # =========================
    if action not in TOOLS:
        return f"❌ 알 수 없는 tool: {action}"

    observation = TOOLS[action].invoke(parsed.get("arguments", {}))

    print("\n[TOOL 실행]")
    print(action)
    print("[OBSERVATION]")
    print(observation)

    # =========================
    # 🔥 2번째 호출 (필요할 때만)
    # =========================
    followup_prompt = f"""
    아래 데이터를 기반으로 최종 답변을 생성해라.

    Tool 결과:
    {observation}

    사용자 질문:
    {user_input}

    자연스럽게 한국어로 답변해라.
    """

    final = llm.invoke(followup_prompt).content

    return final.strip()


# =========================
# 테스트
# =========================
if __name__ == "__main__":
    print(run_agent("방금 뒤에서 누가 지나감"))


[LLM RAW]
```json
{
  "action": "get_vehicle_status",
  "arguments": {
    "location": "뒤쪽" 
  },
  "final_answer": "차량의 현재 위치 정보를 알려드릴 수 있습니다. 어디에서 지나갔는지 확인해 보세요."
}
```

[TOOL 실행]
get_vehicle_status
[OBSERVATION]
현재 속도: 280km/h, 연료: 12.5L
지난번에 혹시 누구를 보았는지 알려주세요! 😊  현재 속도와 연료량을 기준으로, 어떤 사람이 얼마나 빨리 지났는지 추측하기가 쉽습니다. 😉


In [45]:
import re
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", temperature=0)

chat_history = []

@tool
def get_vehicle_status(query: str = "") -> str:
    """차량의 현재 속도와 연료 상태를 확인합니다."""
    return "현재 속도: 280km/h, 연료: 12.5L"

tools = {"get_vehicle_status": get_vehicle_status}

def run_onboard_agent(user_input):
    global chat_history
    
    history_text = ""
    for chat in chat_history:
        history_text += f"User: {chat['user']}\nAgent: {chat['agent']}\n"

    prompt = f"""
    Answer the user's question. You can use the following tool:
    [get_vehicle_status]

    If you need to use the tool, reply EXACTLY like this:
    Action: get_vehicle_status

    If you have the answer, reply EXACTLY like this:
    Final Answer: [your answer]

    you speak korean

    Recent Conversation History:
    {history_text}

    Question: {user_input}
    """
    
    final_answer = "결과 도출 실패"

    for _ in range(3):
        response = llm.invoke(prompt).content
        
        if "Final Answer:" in response:
            final_answer = response.split("Final Answer:")[-1].strip()
            break
            
        if "Action: get_vehicle_status" in response:
            print("Action: get_vehicle_status")
            observation = tools["get_vehicle_status"].invoke({"query": ""})
            prompt += f"\n{response}\nObservation: {observation}\nNow provide the Final Answer:\n"
        else:
            final_answer = response.strip()
            break

    chat_history.append({"user": user_input, "agent": final_answer})
    
    if len(chat_history) > 5:
        chat_history = chat_history[-5:]

    return final_answer

print("첫 번째 질문:", run_onboard_agent("현재속도 알려줘"))
print("두 번째 질문:", run_onboard_agent("방금 내가 속도 말고 또 뭘 물어봤지?"))
print("질문3:", run_onboard_agent("지금까진 한질문은?"))
print("질문4:", run_onboard_agent("너는 무슨 일을 할수 있는가?"))
print("질문5:", run_onboard_agent("일상적인 대화가 가능한가?"))
print("질문6:", run_onboard_agent("내가 4번쨰 한질문은?"))


Action: get_vehicle_status
첫 번째 질문: 현재 속도: 280km/h, 연료: 12.5L
Action: get_vehicle_status
두 번째 질문: 이전에 속도만 물었던 것 같아요.
질문3: 현재 속도: 280km/h, 연료: 12.5L
질문4: 현재 속도: 280km/h, 연료: 12.5L
질문5: 네, 일상적인 대화가 가능합니다.
질문6: 현재 속도: 280km/h, 연료: 12.5L


In [46]:
import re
from langchain_ollama import ChatOllama
from langchain_core.tools import tool

llm = ChatOllama(model="hf.co/bartowski/gemma-2-2b-it-abliterated-GGUF:Q4_K_M", temperature=0)

chat_history = []

@tool
def get_vehicle_status(query: str = "") -> str:
    """차량의 현재 속도와 연료 상태를 확인합니다."""
    return "현재 속도: 280km/h, 연료: 12.5L"

tools = {"get_vehicle_status": get_vehicle_status}

def run_onboard_agent(user_input):
    global chat_history
    
    # 1. 대화 기록을 보기 좋게 정리
    history_text = ""
    if not chat_history:
        history_text = "None"
    else:
        for idx, chat in enumerate(chat_history):
            history_text += f"[Turn {idx+1}] User: {chat['user']} -> Agent: {chat['agent']}\n"

    # 2. 프롬프트 구조화 (시스템 지시 / 기록 / 현재 질문을 명확히 분리)
    prompt = f"""You are a smart AI assistant for a racing game.

[Instructions]
1. Read the "Recent Conversation History" and the "Current Question".
2. If the user asks about the past conversation, answer based ONLY on the "Recent Conversation History". Do not use tools.
3. If the user asks about the CURRENT vehicle status, use the tool.
4. When answering, provide a natural response in Korean. Do not just copy the tool's raw data.

[Tool Available]
- get_vehicle_status: Use this ONLY when asked about current speed or fuel.
To use the tool, reply exactly: Action: get_vehicle_status

[Recent Conversation History]
{history_text}

[Current Question]
{user_input}

To provide the final answer, reply exactly:
Final Answer: [Your natural Korean response]
"""
    
    final_answer = "결과 도출 실패"

    for step in range(3):
        response = llm.invoke(prompt).content
        
        # 모델이 'Final Answer:' 형식을 지켰을 때
        if "Final Answer:" in response:
            final_answer = response.split("Final Answer:")[-1].strip()
            break
            
        # 모델이 도구를 쓰겠다고 했을 때
        elif "Action: get_vehicle_status" in response:
            print(">> [도구 실행됨: get_vehicle_status]")
            observation = tools["get_vehicle_status"].invoke({"query": ""})
            # 도구 결과를 프롬프트에 추가하고 다시 질문
            prompt += f"\nAction: get_vehicle_status\nObservation: {observation}\nNow synthesize the observation and provide the Final Answer naturally:\n"
            
        # 모델이 형식을 무시하고 그냥 대답했을 때 (Fallback)
        else:
            final_answer = response.strip()
            # "Action:" 이나 "Final Answer:" 같은 시스템 텍스트가 섞여있으면 제거
            final_answer = re.sub(r'(Action:|Final Answer:|Observation:).*', '', final_answer, flags=re.IGNORECASE).strip()
            if final_answer: 
                break

    # 3. 대화 기록 저장 및 슬라이딩 윈도우
    if final_answer and final_answer != "결과 도출 실패":
        chat_history.append({"user": user_input, "agent": final_answer})
        if len(chat_history) > 5:
            chat_history = chat_history[-5:]

    return final_answer

print("1:", run_onboard_agent("현재속도 알려줘"))
print("2:", run_onboard_agent("방금 내가 속도 말고 또 뭘 물어봤지?"))
print("3:", run_onboard_agent("지금까지 한 질문들 요약해줄래?"))
print("4:", run_onboard_agent("너는 무슨 일을 할 수 있는가?"))
print("5:", run_onboard_agent("일상적인 대화가 가능한가?"))
print("6:", run_onboard_agent("내가 4번째 한 질문이 뭐였지?"))

1: 현재 속도는 어디에 있는지 확인해 주세요. 🏎️💨
2: 어떤 것들을 궁금해하셨는지 알려주세요! 😊
3: 이전에 궁금했던 질문은 "현재 속도 알려줘" 입니다. 😊 🏎️💨
4: 저는 현재 속도나 연료 수량 등 차량 상태를 알려드릴 수 있어요! 😊 🏎️💨
5: 네, 일상적인 대화를 할 수 있습니다. 😄  무엇이 궁금하세요? 😊 🏎️💨
6: 네, 이전에 궁금했던 질문은 "현재 속도 알려줘" 입니다. 😊 🏎️💨
